# Notebook 06 - SQL Analytics using Star Schema

In [2]:
import duckdb
import pandas as pd

In [3]:
con = duckdb.connect()

con.execute("""
CREATE OR REPLACE TABLE dim_city AS
SELECT * FROM read_csv_auto('../data/curated/dim_city.csv')
""")

con.execute("""
CREATE OR REPLACE TABLE dim_neighbourhood AS
SELECT * FROM read_csv_auto('../data/curated/dim_neighbourhood.csv')
""")

con.execute("""
CREATE OR REPLACE TABLE dim_listing AS
SELECT * FROM read_csv_auto('../data/curated/dim_listing.csv')
""")

con.execute("""
CREATE OR REPLACE TABLE fact_listing AS
SELECT * FROM read_csv_auto('../data/curated/fact_listing.csv')
""")

In [4]:
con.sql("""
SELECT
    dc.city,
    ROUND(AVG(fl.price_clean), 2) AS avg_price,
    ROUND(AVG(fl.occupancy_rate), 3) AS avg_occupancy_rate,
    COUNT(*) AS listing_count
FROM fact_listing fl
JOIN dim_city dc
ON fl.city_id = dc.city_id
GROUP BY dc.city
ORDER BY avg_price DESC
""").df()

,city,avg_price,avg_occupancy_rate,listing_count
0,New York,255.00,0.570,35036
1,London,229.92,0.603,96871


In [5]:
con.sql("""
SELECT
    dc.city,
    dn.neighbourhood_cleansed,
    ROUND(MEDIAN(fl.price_clean), 2) AS median_price,
    COUNT(*) AS listing_count
FROM fact_listing fl
JOIN dim_city dc
    ON fl.city_id = dc.city_id
JOIN dim_neighbourhood dn
    ON fl.neighbourhood_id = dn.neighbourhood_id
WHERE fl.price_clean IS NOT NULL
GROUP BY dc.city, dn.neighbourhood_cleansed
HAVING COUNT(*) >= 50
ORDER BY median_price DESC
LIMIT 10
""").df()

,city,neighbourhood_cleansed,median_price,listing_count
0,New York,Tribeca,525.83,69
1,New York,SoHo,431.10,151
2,New York,Financial District,397.61,293
3,New York,Greenwich Village,385.98,102
4,New York,Theater District,344.00,269
5,New York,Midtown,338.61,1362
6,New York,West Village,287.82,182
7,New York,Carroll Gardens,258.10,65
8,New York,Greenpoint,254.43,246
9,New York,Murray Hill,254.43,327


In [6]:
con.sql("""
SELECT
    dc.city,
    ROUND(AVG(fl.occupancy_rate), 3) AS avg_occupancy_rate,
    ROUND(AVG(fl.availability_rate), 3) AS avg_availability_rate
FROM fact_listing fl
JOIN dim_city dc
    ON fl.city_id = dc.city_id
GROUP BY dc.city
ORDER BY avg_occupancy_rate DESC
""").df()

,city,avg_occupancy_rate,avg_availability_rate
0,London,0.603,0.397
1,New York,0.570,0.430


In [7]:
con.sql("""
SELECT
    dc.city,
    dl.room_type,
    ROUND(AVG(fl.price_clean), 2) AS avg_price,
    ROUND(MEDIAN(fl.price_clean), 2) AS median_price,
    COUNT(*) AS listing_count
FROM fact_listing fl
JOIN dim_city dc
    ON fl.city_id = dc.city_id
JOIN dim_listing dl
    ON fl.listing_id = dl.listing_id
WHERE fl.price_clean IS NOT NULL
GROUP BY dc.city, dl.room_type
ORDER BY dc.city, median_price DESC
""").df()

,city,room_type,avg_price,median_price,listing_count
0,London,Hotel room,657.83,281.00,72
1,London,Entire home/apt,279.35,175.00,42318
2,London,Private room,121.71,61.00,19382
3,London,Shared room,96.91,32.00,191
4,New York,Hotel room,814.35,529.00,332
5,New York,Entire home/apt,297.97,205.41,11003
6,New York,Private room,184.76,100.86,9164
7,New York,Shared room,178.02,66.41,194


In [8]:
con.sql("""
SELECT
    dc.city,
    fl.listing_id,
    dl.room_type,
    dl.property_type,
    fl.total_reviews,
    ROUND(fl.price_clean, 2) AS price_clean,
    ROUND(fl.occupancy_rate, 3) AS occupancy_rate
FROM fact_listing fl
JOIN dim_city dc
    ON fl.city_id = dc.city_id
JOIN dim_listing dl
    ON fl.listing_id = dl.listing_id
WHERE fl.total_reviews IS NOT NULL
ORDER BY fl.total_reviews DESC
LIMIT 10
""").df()

,city,listing_id,room_type,property_type,total_reviews,price_clean,occupancy_rate
0,New York,858697692672545141,Private room,Private room in bed and breakfast,4310.0,99.00,0.005
1,New York,51619634,Private room,Room in hotel,2300.0,305.50,0.008
2,New York,691676460109271194,Entire home/apt,Entire loft,2276.0,140.00,0.044
3,New York,37122502,Private room,Room in boutique hotel,1998.0,250.00,0.156
4,London,47408549,Private room,Room in hotel,1902.0,162.00,0.008
5,London,43120947,Private room,Room in hotel,1647.0,114.00,0.036
6,New York,992970965790772607,Private room,Room in hotel,1484.0,185.56,0.148
7,London,19670926,Entire home/apt,Entire serviced apartment,1443.0,240.00,0.249
8,New York,700888293266225258,Entire home/apt,Entire loft,1337.0,140.00,0.047
9,New York,53132475,Private room,Room in hotel,1231.0,277.00,0.014
